# 01. База знаний КР и сравнение вариантов поиска

КР 154_4 «Острый коронарный синдром без подъема сегмента ST электрокардиограммы» (Минздрав РФ, 2024) → структурированные чанки → три варианта поиска: **BM25** (лемматизация pymorphy3 + раскрытие сокращений), **dense** (`deepvk/USER2-base`), **hybrid** (RRF).

> Colab: `!git clone <url>`, `%cd AI_Healthcare`, `!pip install -e .`. GPU не нужен: эмбеддинги 383 чанков уже посчитаны в `data/kb/`, на CPU кодируются только запросы.

In [1]:
import os, warnings
os.environ['TQDM_DISABLE'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
warnings.filterwarnings('ignore')
import transformers
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()
from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # без предупреждения о запросах к HF Hub без токена

In [2]:
import pandas as pd
from collections import Counter
from resident.kb.parse import load_chunks

chunks = load_chunks()
print(len(chunks), 'чанков:', dict(Counter(c.kind for c in chunks)))
pd.Series(Counter(c.uur + c.udd for c in chunks if c.kind == 'thesis')).sort_index().rename('тезисов по УУР+УДД')

383 чанков: {'text': 79, 'thesis': 219, 'criterion': 19, 'table': 66}


A1     29
A2     27
A3      1
B1      1
B2     35
B3      5
C2      1
C4     14
C5    106
Name: тезисов по УУР+УДД, dtype: int64

## Пример чанка
Тезис хранит пункт КР, порядковый номер тезиса, УУР/УДД и комментарий - этого хватает для ссылки вида `[КР 154_4, п. 3.2.3.2, тезис 6; УУР A, УДД 2]`.

In [3]:
kb = {c.chunk_id: c for c in chunks}
c = kb['154_4:3.2.3.2:t06']
print(c.citation)
print(c.model_dump_json(indent=1)[:1200])

[КР 154_4, п. 3.2.3.2, тезис 6; УУР A, УДД 2]
{
 "chunk_id": "154_4:3.2.3.2:t06",
 "cr_id": "154_4",
 "kind": "thesis",
 "section_num": "3.2.3.2",
 "section_path": [
  "3. Лечение",
  "3.2. Медикаментозное лечение заболевания",
  "3.2.3. Антитромботическая терапия",
  "3.2.3.2. Парентеральные антикоагулянты"
 ],
 "thesis_idx": 6,
 "locator": "тезис 6",
 "text": "Если в ранние сроки лечения ОКСбпST не планируется выполнение КГ, для снижения риска ишемических событий рекомендуется фондапаринукс натрия (2,5 мг подкожно ежедневно) как имеющий наиболее благоприятный профиль эффективности и безопасности, если к нему нет противопоказаний [218, 219, 222].",
 "comment": "Больному, получающему фондапаринукс натрия, во время ЧКВ следует внутривенно вводить НФГ – первоначальный болюс в дозе 85 МЕ/кг; необходимость повторных болюсов определяется величиной АВС крови. Фондапаринукс натрия противопоказан при рСКФ ниже 20 мл/мин/1,73 м 2 . Если фондапаринукс натрия недоступен, может использоваться энок

In [4]:
row = kb['154_4:А3:tab05.r10']  # большие таблицы дозирования разбиты по строкам
print(row.citation)
print(row.text)

[КР 154_4, п. А3, табл. 5, строка 10]
Дозы антитромботических лекарственных средств при нарушенной функии почек
Препарат | рСКФ 30-59 мл/мин/1,73 м2 или КлКр 30–59 мл/мин | рСКФ 15-29 мл/мин/1,73 м2 или КлКр 15–29 мл/мин | рСКФ < 15 мл/мин/1,73 м2 или КлКр < 15 мл/мин
Антикоагулянты
Фондапаринукс натрия 1 | Не рекомендован у пациентов с КлКр <20 мл/мин. Коррекция дозы не требуется у пациентов с КлКр ≥20 мл/мин.


## Сравнение поиска
Эталон: 30 перефразированных клинических запросов → релевантные `chunk_id` (`data/eval/retrieval_gold.jsonl`).

In [5]:
from resident.retrieval import build_retriever
from resident.retrieval.evaluate import evaluate, load_gold

gold = load_gold()
retrievers = {k: build_retriever(k, chunks) for k in ['bm25', 'dense', 'hybrid']}
results, per_query = {}, {}
for name, r in retrievers.items():
    results[name], per_query[name] = evaluate(r, gold)
pd.DataFrame(results).T

,Recall@5,Recall@10,MRR@10,nDCG@5
bm25,0.778,0.861,0.584,0.602
dense,0.817,0.850,0.665,0.691
hybrid,0.861,0.961,0.691,0.692


## Где варианты расходятся (Recall@5 по запросам)

In [6]:
cmp = pd.DataFrame({k: v.set_index('query')['Recall@5'] for k, v in per_query.items()})
cmp[cmp.nunique(axis=1) > 1]

,bm25,dense,hybrid
query,,,
как подтвердить инфаркт по динамике тропонина,1.000000,0.5,0.500000
"свёртываемость крови: АЧТВ, МНО перед лечением",1.000000,0.0,1.000000
в какие сроки делать коронарографию пациенту высокого риска,0.500000,0.0,1.000000
нагрузочная и поддерживающая доза аспирина,0.500000,1.0,0.500000
двойная антитромбоцитарная терапия на 12 месяцев,0.000000,1.0,1.000000
чем заменить тикагрелор при противопоказаниях,0.500000,0.0,0.500000
аспирин при тромбоцитопении,0.000000,1.0,1.000000
переход на клопидогрел для снижения риска кровотечений,0.000000,1.0,1.000000
антикоагулянт если инвазивное лечение не планируется,1.000000,0.0,0.000000


## Фильтр по разделу КР
Planner ищет в пределах раздела своей клинической области - это отсекает похожие, но нерелевантные пункты.

In [7]:
for h in retrievers['hybrid'].search('антикоагулянт при тяжёлой почечной недостаточности', k=5, sections=['3.2.3.2', 'А3']):
    print(h.rank, h.chunk.citation, '|', h.chunk.text.splitlines()[-1][:100])

1 [КР 154_4, п. А3, табл. 5, строка 9] | Эноксапарин натрия | Обычная доза | Согласно инструкции, при тяжелой почечной недостаточности (без у
2 [КР 154_4, п. А3, табл. 5, строка 3] | Прасугрел | Обычная доза | Обычная доза | Опыт применения прасугрела у пациентов с почечной недостат
3 [КР 154_4, п. 3.2.3.2, тезис 1; УУР A, УДД 1] | У пациентов с подтвержденным диагнозом ОКСбпST рекомендуется парентеральное введение антикоагулянтов
4 [КР 154_4, п. 3.2.3.2, тезис 2; УУР C, УДД 5] | После успешного ЧКВ у пациентов с ОКСбпST рекомендуется прекратить парентеральное введение антикоагу
5 [КР 154_4, п. А3, табл. 5, строка 4] | Тикагрелор 3 | Обычная доза | Обычная доза | Отсутствует информация о применении препарата у пациент


**Вывод:** гибрид даёт лучший Recall@10 (почти все релевантные пункты попадают в контекст LLM) и лучший MRR - он используется по умолчанию (`RETRIEVER=hybrid`).